# G02 — P02 : Régularisation et Généralisation
## Notebook d'analyse des résultats Optuna
**Dataset** : IMDb (D01) | **Modèle** : BERT-base (M02) | **Méthode** : Optuna Grid Search

Ce notebook permet d'explorer interactivement les résultats du Grid Search Optuna
sur les hyperparamètres de régularisation (weight_decay × dropout).

Exécutez d'abord `python main.py` pour générer les fichiers `results/`.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Style uniforme
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Imports OK')

## 1. Chargement des résultats Optuna

In [ ]:
# Résultats synthétiques (CSV)
df = pd.read_csv('../results/optuna_results.csv')
df['weight_decay_str'] = df['weight_decay'].apply(lambda x: f'{float(x):.0e}')

print(f'Nombre de trials : {len(df)}')
print('\nTop 5 combinaisons :')
df.sort_values('val_f1', ascending=False).head(5).style.background_gradient(cmap='YlOrRd', subset=['val_f1'])

## 2. Heatmap : weight_decay × dropout → val_F1

In [ ]:
pivot = df.pivot_table(
    index='dropout',
    columns='weight_decay_str',
    values='val_f1',
    aggfunc='max'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'F1-score (validation)'})
ax.set_title('G02 — P02 : Heatmap des performances\n(BERT-base, IMDb)', fontsize=14, fontweight='bold')
ax.set_xlabel('Weight Decay')
ax.set_ylabel('Dropout Rate')
plt.tight_layout()
plt.show()

## 3. Courbes de convergence (meilleur trial)

In [ ]:
with open('../results/optuna_details.json') as f:
    details = json.load(f)

# Tri par val_f1
details_sorted = sorted(details, key=lambda t: t['val_f1'], reverse=True)
best = details_sorted[0]

h = best['history']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Meilleur trial — wd={best['params']['weight_decay']:.0e} "
    f"| dropout={best['params']['dropout']} | val_F1={best['val_f1']:.4f}",
    fontsize=13, fontweight='bold'
)

# Accuracy
axes[0].plot(h['epoch'], h['train_acc'], 'b-o', label='Train Accuracy', lw=2)
axes[0].plot(h['epoch'], h['val_acc'],   'r--s', label='Val Accuracy', lw=2)
axes[0].fill_between(h['epoch'], h['train_acc'], h['val_acc'],
                     alpha=0.2, color='orange', label='Overfitting gap')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Courbe de convergence')
axes[0].legend()

# Loss
axes[1].plot(h['epoch'], h['gap'], 'o-', color='darkorange', lw=2)
axes[1].axhline(0, color='gray', lw=0.8, ls=':')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Gap (train_acc - val_acc)')
axes[1].set_title('Overfitting gap par époque')

plt.tight_layout()
plt.show()

## 4. Analyse du gap d'overfitting (tous les trials)

In [ ]:
rows = []
for t in details:
    h = t['history']
    final_gap = h['gap'][-1] if h.get('gap') else None
    rows.append({
        'weight_decay': t['params']['weight_decay'],
        'dropout':      t['params']['dropout'],
        'final_gap':    final_gap,
        'val_f1':       t['val_f1'],
    })
df_gap = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 6))
for dp in sorted(df_gap['dropout'].unique()):
    sub = df_gap[df_gap['dropout'] == dp].sort_values('weight_decay')
    ax.plot(sub['weight_decay'], sub['final_gap'],
            marker='o', lw=2, label=f'dropout={dp}')

ax.set_xscale('log')
ax.set_xlabel('Weight Decay (log scale)', fontsize=12)
ax.set_ylabel('Gap final (train_acc − val_acc)', fontsize=12)
ax.set_title('Influence du weight decay sur l\'overfitting\npar niveau de dropout', fontsize=13)
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nCorrélation gap ↔ val_F1 :')
print(f"  Pearson r = {df_gap['final_gap'].corr(df_gap['val_f1']):.4f}")
print('  (r négatif = plus le gap est faible, plus le F1 est élevé)')

## 5. Tableau de synthèse final

In [ ]:
df_summary = df.sort_values('val_f1', ascending=False).reset_index(drop=True)
df_summary.index += 1
df_summary['weight_decay'] = df_summary['weight_decay'].apply(lambda x: f'{float(x):.0e}')

print('='*55)
print(' G02 — P02 : Classement des 12 combinaisons Optuna')
print('='*55)
display(
    df_summary[['weight_decay', 'dropout', 'val_f1', 'elapsed_s']]
    .rename(columns={'weight_decay': 'Weight Decay', 'dropout': 'Dropout',
                     'val_f1': 'Val F1', 'elapsed_s': 'Durée (s)'})
    .style
    .background_gradient(cmap='YlOrRd', subset=['Val F1'])
    .format({'Val F1': '{:.4f}'})
)

best = df_summary.iloc[0]
print(f"\n🏆 Meilleur : weight_decay={best['weight_decay']} | dropout={best['dropout']} | F1={best['val_f1']:.4f}")